# Step 4 - import json files & combine 

In [1]:
import pandas as pd
import json
import re
import os

## 1. Import all json files from `data/landing/domain_data/`, then combine together as a pd dataframe

In [2]:
domain_data_directory = '../data/landing/domain_data/'  # Specify the directory to traverse

domain_data = pd.DataFrame()
# Iterate through all json files in the directory
for filename in os.listdir(domain_data_directory):
    if filename.endswith('.json'):  # check if the file is json file
        file_path = os.path.join(domain_data_directory, filename)
        
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
        
    data = pd.DataFrame.from_dict(data).T
    domain_data = pd.concat([domain_data, data], ignore_index=True)

domain_data

,postcode,name,cost_text,coordinates,rooms,parking,desc
0,3149,105/436-442 Huntingdale Road Mount Waverley VI...,$525.00,"[-37.8832957, 145.1090502]","[2 Beds, 1 Bath]",[1 Parking],Sienna Simpson
1,3149,4A Adrienne Crescent Mount Waverley VIC 3149,"$1,250","[-37.8956779, 145.1171243]","[4 Beds, 3 Baths]",[2 Parking],Matthew Swinnerton
2,3149,1/2 Grenfell Rd Mount Waverley VIC 3149,$500,"[-37.8749788, 145.1117625]","[2 Beds, 1 Bath]",[1 Parking],"data-testid=""nbn-connection_description""><spa..."
3,3149,310/436-442 Huntingdale Road Mount Waverley VI...,$575.00,"[-37.8832957, 145.1090502]","[2 Beds, 1 Bath]",[2 Parking],Register now to view this stunning two bedroom...
4,3149,2/8 Bennett Avenue Mount Waverley VIC 3149,$530,"[-37.860802, 145.148173]","[3 Beds, 1 Bath]",[2 Parking],PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT
...,...,...,...,...,...,...,...
8958,3081,1/19 Law Street Heidelberg Heights VIC 3081,$550 Per Week,"[-37.7449638, 145.0484437]","[3 Beds, 1 Bath]",[2 Parking],"class=""css-dxogle"">* Unverified feature<svg a..."
8959,3081,64 Tobruk Avenue Heidelberg West VIC 3081,$500.00,"[-37.7472529, 145.038464]","[3 Beds, 1 Bath]",[2 Parking],"class=""css-dxogle"">* Unverified feature<svg a..."
8960,3081,8 Disney Street Heidelberg Heights VIC 3081,$680 Per Week,"[-37.7493478, 145.0478948]","[3 Beds, 3 Baths]",[2 Parking],"class=""css-dxogle"">* Unverified feature<svg a..."
8961,3081,4B Katoomba Court Heidelberg West VIC 3081,$600 PW / $2607 PCM,"[-37.73547670000001, 145.0376889]","[3 Beds, 1 Bath]",[2 Parking],"class=""css-dxogle"">* Unverified feature<svg a..."


Store the pd df as `domain_data` into `data/raw/domain_data/`

In [3]:
'''
output_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/test/domain_data_raw.csv'
domain_data.to_csv(output_directory, index=False)
'''

"\noutput_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/test/domain_data_raw.csv'\ndomain_data.to_csv(output_directory, index=False)\n"

In [4]:
output_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/raw/domain_data/domain_data.csv'
domain_data.to_csv(output_directory, index=False)

## 2. preprocessing for raw `domain_data`

In [5]:
domain_data = pd.read_csv('../data/raw/domain_data/domain_data.csv')

### 2.1 Overview of `domain_data`

In [6]:
domain_data

,postcode,name,cost_text,coordinates,rooms,parking,desc
0,3149.0,105/436-442 Huntingdale Road Mount Waverley VI...,$525.00,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",['1 Parking'],Sienna Simpson
1,3149.0,4A Adrienne Crescent Mount Waverley VIC 3149,"$1,250","[-37.8956779, 145.1171243]","['4 Beds', '3 Baths']",['2 Parking'],Matthew Swinnerton
2,3149.0,1/2 Grenfell Rd Mount Waverley VIC 3149,$500,"[-37.8749788, 145.1117625]","['2 Beds', '1 Bath']",['1 Parking'],"data-testid=""nbn-connection_description""><spa..."
3,3149.0,310/436-442 Huntingdale Road Mount Waverley VI...,$575.00,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",['2 Parking'],Register now to view this stunning two bedroom...
4,3149.0,2/8 Bennett Avenue Mount Waverley VIC 3149,$530,"[-37.860802, 145.148173]","['3 Beds', '1 Bath']",['2 Parking'],PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT
...,...,...,...,...,...,...,...
8958,3081.0,1/19 Law Street Heidelberg Heights VIC 3081,$550 Per Week,"[-37.7449638, 145.0484437]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8959,3081.0,64 Tobruk Avenue Heidelberg West VIC 3081,$500.00,"[-37.7472529, 145.038464]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8960,3081.0,8 Disney Street Heidelberg Heights VIC 3081,$680 Per Week,"[-37.7493478, 145.0478948]","['3 Beds', '3 Baths']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8961,3081.0,4B Katoomba Court Heidelberg West VIC 3081,$600 PW / $2607 PCM,"[-37.73547670000001, 145.0376889]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."


### 2.2 preprocessing

#### 2.2.1 extract info from 'cost_text' to create the feature 'rental_price'

extract 'rental_price' & rename

In [7]:
domain_data['cost_text'] = domain_data['cost_text'].str.extract(r'(\d+)')
domain_data.rename(columns={'cost_text': 'rental_price'}, inplace=True)

change 'rental_price' from string type to float type & drop NaN

In [8]:
domain_data['rental_price'] = domain_data['rental_price'].astype(float)
domain_data = domain_data.dropna(subset=['rental_price'])
domain_data

,postcode,name,rental_price,coordinates,rooms,parking,desc
0,3149.0,105/436-442 Huntingdale Road Mount Waverley VI...,525.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",['1 Parking'],Sienna Simpson
1,3149.0,4A Adrienne Crescent Mount Waverley VIC 3149,1.0,"[-37.8956779, 145.1171243]","['4 Beds', '3 Baths']",['2 Parking'],Matthew Swinnerton
2,3149.0,1/2 Grenfell Rd Mount Waverley VIC 3149,500.0,"[-37.8749788, 145.1117625]","['2 Beds', '1 Bath']",['1 Parking'],"data-testid=""nbn-connection_description""><spa..."
3,3149.0,310/436-442 Huntingdale Road Mount Waverley VI...,575.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",['2 Parking'],Register now to view this stunning two bedroom...
4,3149.0,2/8 Bennett Avenue Mount Waverley VIC 3149,530.0,"[-37.860802, 145.148173]","['3 Beds', '1 Bath']",['2 Parking'],PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT
...,...,...,...,...,...,...,...
8958,3081.0,1/19 Law Street Heidelberg Heights VIC 3081,550.0,"[-37.7449638, 145.0484437]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8959,3081.0,64 Tobruk Avenue Heidelberg West VIC 3081,500.0,"[-37.7472529, 145.038464]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8960,3081.0,8 Disney Street Heidelberg Heights VIC 3081,680.0,"[-37.7493478, 145.0478948]","['3 Beds', '3 Baths']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."
8961,3081.0,4B Katoomba Court Heidelberg West VIC 3081,600.0,"[-37.73547670000001, 145.0376889]","['3 Beds', '1 Bath']",['2 Parking'],"class=""css-dxogle"">* Unverified feature<svg a..."


we found a rental price which is bigger than $1\times10^8$, so we made a 5000 restriction

In [9]:
print(domain_data['rental_price'].describe())

count    8.810000e+03
mean     5.476971e+04
std      5.090429e+06
min      1.000000e+00
25%      4.300000e+02
50%      5.200000e+02
75%      6.400000e+02
max      4.777964e+08
Name: rental_price, dtype: float64


In [10]:
domain_data = domain_data[domain_data['rental_price'] <= 5000]

double check

In [11]:
print(domain_data['rental_price'].describe())

count    8809.000000
mean      536.356454
std       235.267908
min         1.000000
25%       430.000000
50%       520.000000
75%       640.000000
max      5000.000000
Name: rental_price, dtype: float64


#### 2.2.2 extract info from 'parking' to create the feature 'num_parking'

In [12]:
value_counts = domain_data['parking'].value_counts(dropna=False)
print(value_counts)

['1 Parking']     3741
['2 Parking']     2953
['− Parking']     1670
['3 Parking']      229
['4 Parking']      145
['6 Parking']       21
['5 Parking']       19
[]                   7
['10 Parking']       5
['7 Parking']        5
['8 Parking']        5
['9 Parking']        3
['11 Parking']       3
['14 Parking']       1
['12 Parking']       1
['20 Parking']       1
Name: parking, dtype: int64


define a function to extract the info from 'parking'

In [13]:
def replace_list_value(parking_list):
    if '− Parking' in parking_list:
        return 0
    match = re.search(r'(\d+) Parking', parking_list)
    if match:
        return int(match.group(1)) 


use the function to extract & rename & drop NaN

In [14]:
domain_data['parking'] = domain_data['parking'].apply(replace_list_value)

domain_data.rename(columns={'parking': 'num_parking'}, inplace=True)

domain_data = domain_data.dropna(subset=['num_parking'])

/tmp/ipykernel_705/394989124.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  domain_data['parking'] = domain_data['parking'].apply(replace_list_value)
/tmp/ipykernel_705/394989124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  domain_data.rename(columns={'parking': 'num_parking'}, inplace=True)


double check

In [15]:
value_counts = domain_data['num_parking'].value_counts(dropna=False)
print(value_counts)
domain_data.head()

1.0     3741
2.0     2953
0.0     1670
3.0      229
4.0      145
6.0       21
5.0       19
10.0       5
7.0        5
8.0        5
9.0        3
11.0       3
14.0       1
12.0       1
20.0       1
Name: num_parking, dtype: int64


,postcode,name,rental_price,coordinates,rooms,num_parking,desc
0,3149.0,105/436-442 Huntingdale Road Mount Waverley VI...,525.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",1.0,Sienna Simpson
1,3149.0,4A Adrienne Crescent Mount Waverley VIC 3149,1.0,"[-37.8956779, 145.1171243]","['4 Beds', '3 Baths']",2.0,Matthew Swinnerton
2,3149.0,1/2 Grenfell Rd Mount Waverley VIC 3149,500.0,"[-37.8749788, 145.1117625]","['2 Beds', '1 Bath']",1.0,"data-testid=""nbn-connection_description""><spa..."
3,3149.0,310/436-442 Huntingdale Road Mount Waverley VI...,575.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",2.0,Register now to view this stunning two bedroom...
4,3149.0,2/8 Bennett Avenue Mount Waverley VIC 3149,530.0,"[-37.860802, 145.148173]","['3 Beds', '1 Bath']",2.0,PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT


#### 2.2.3 extract info from 'rooms' to create the feature 'num_bedroom' & 'num_bathroom'

In [16]:
value_counts = domain_data['rooms'].value_counts(dropna=False)
print(value_counts)

['2 Beds', '1 Bath']       1869
['1 Bed', '1 Bath']        1438
['3 Beds', '2 Baths']      1411
['4 Beds', '2 Baths']      1337
['3 Beds', '1 Bath']       1036
['2 Beds', '2 Baths']       992
['4 Beds', '3 Baths']       205
['4 Beds', '1 Bath']         94
['3 Beds', '3 Baths']        91
['5 Beds', '2 Baths']        81
['0 Beds', '1 Bath']         66
['5 Beds', '3 Baths']        52
['0 Beds', '0 Baths']        29
['4 Beds', '4 Baths']        21
['5 Beds', '1 Bath']         12
['5 Beds', '4 Baths']        11
['6 Beds', '2 Baths']        11
['1 Bed', '2 Baths']         10
['2 Beds', '3 Baths']         5
['8 Beds', '3 Baths']         5
['6 Beds', '3 Baths']         4
['5 Beds', '5 Baths']         3
['9 Beds', '2 Baths']         2
['1 Bed', '3 Baths']          2
['1 Bed', '0 Baths']          2
['7 Beds', '2 Baths']         2
['6 Beds', '4 Baths']         2
['3 Beds', '23 Baths']        1
['9 Beds', '1 Bath']          1
['4 Beds', '5 Baths']         1
['7 Beds', '1 Bath']          1
['10 Bed

define a function to extract the bedroom info from 'rooms'

In [17]:
def replace_list_value(room_list):
    match = re.search(r'(\d+) Bed', room_list)
    if match:
        return int(match.group(1)) 

use the function to extract

In [18]:
domain_data['num_bedroom'] = domain_data['rooms'].apply(replace_list_value)

double check

In [19]:
value_counts = domain_data['num_bedroom'].value_counts(dropna=False)
print(value_counts)
domain_data.head()

2     2866
3     2539
4     1658
1     1453
5      159
0       96
6       17
8        6
7        3
9        3
10       2
Name: num_bedroom, dtype: int64


,postcode,name,rental_price,coordinates,rooms,num_parking,desc,num_bedroom
0,3149.0,105/436-442 Huntingdale Road Mount Waverley VI...,525.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",1.0,Sienna Simpson,2
1,3149.0,4A Adrienne Crescent Mount Waverley VIC 3149,1.0,"[-37.8956779, 145.1171243]","['4 Beds', '3 Baths']",2.0,Matthew Swinnerton,4
2,3149.0,1/2 Grenfell Rd Mount Waverley VIC 3149,500.0,"[-37.8749788, 145.1117625]","['2 Beds', '1 Bath']",1.0,"data-testid=""nbn-connection_description""><spa...",2
3,3149.0,310/436-442 Huntingdale Road Mount Waverley VI...,575.0,"[-37.8832957, 145.1090502]","['2 Beds', '1 Bath']",2.0,Register now to view this stunning two bedroom...,2
4,3149.0,2/8 Bennett Avenue Mount Waverley VIC 3149,530.0,"[-37.860802, 145.148173]","['3 Beds', '1 Bath']",2.0,PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT,3


define a function to extract the bathroom info from 'rooms'

In [20]:
def replace_list_value(room_list):
    match = re.search(r'(\d+) Bath', room_list)
    if match:
        return int(match.group(1)) 

use the function to extract

In [21]:
domain_data['num_bathroom'] = domain_data['rooms'].apply(replace_list_value)

double check

In [22]:
value_counts = domain_data['num_bathroom'].value_counts(dropna=False)
print(value_counts)

1     4517
2     3848
3      365
4       35
0       31
5        4
10       1
23       1
Name: num_bathroom, dtype: int64


Delete the feature 'rooms' after extracting the information

In [23]:
domain_data.drop('rooms', axis=1, inplace=True)

Ovewview of `domain_data` after preprocessing

In [24]:
domain_data

,postcode,name,rental_price,coordinates,num_parking,desc,num_bedroom,num_bathroom
0,3149.0,105/436-442 Huntingdale Road Mount Waverley VI...,525.0,"[-37.8832957, 145.1090502]",1.0,Sienna Simpson,2,1
1,3149.0,4A Adrienne Crescent Mount Waverley VIC 3149,1.0,"[-37.8956779, 145.1171243]",2.0,Matthew Swinnerton,4,3
2,3149.0,1/2 Grenfell Rd Mount Waverley VIC 3149,500.0,"[-37.8749788, 145.1117625]",1.0,"data-testid=""nbn-connection_description""><spa...",2,1
3,3149.0,310/436-442 Huntingdale Road Mount Waverley VI...,575.0,"[-37.8832957, 145.1090502]",2.0,Register now to view this stunning two bedroom...,2,1
4,3149.0,2/8 Bennett Avenue Mount Waverley VIC 3149,530.0,"[-37.860802, 145.148173]",2.0,PHONE CARMEN MOSS ON 0412 74 99 30 TO INSPECT,3,1
...,...,...,...,...,...,...,...,...
8958,3081.0,1/19 Law Street Heidelberg Heights VIC 3081,550.0,"[-37.7449638, 145.0484437]",2.0,"class=""css-dxogle"">* Unverified feature<svg a...",3,1
8959,3081.0,64 Tobruk Avenue Heidelberg West VIC 3081,500.0,"[-37.7472529, 145.038464]",2.0,"class=""css-dxogle"">* Unverified feature<svg a...",3,1
8960,3081.0,8 Disney Street Heidelberg Heights VIC 3081,680.0,"[-37.7493478, 145.0478948]",2.0,"class=""css-dxogle"">* Unverified feature<svg a...",3,3
8961,3081.0,4B Katoomba Court Heidelberg West VIC 3081,600.0,"[-37.73547670000001, 145.0376889]",2.0,"class=""css-dxogle"">* Unverified feature<svg a...",3,1


## 3. Store the `domain_data` into `data/curated/domain_data/`

In [25]:
'''
output_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/test/domain_data_test.csv'
domain_data.to_csv(output_directory, index=False)
'''

"\noutput_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/test/domain_data_test.csv'\ndomain_data.to_csv(output_directory, index=False)\n"

In [26]:
output_directory = '../../real-estate-industry-project-open-source-industry-project-22/data/curated/domain_data/domain_data.csv'
domain_data.to_csv(output_directory, index=False)